import all necessary libraries

In [0]:
import pyspark.sql.functions as F

check the external location

In [0]:
%sql
describe external location `control_datalake_raw`

Extract the URL.

In [0]:
raw_dir = spark.sql("describe external location `control_datalake_raw`").select("url").collect()[0][0]
#display(base_dir)

Extract the data from containers to variables

In [0]:
# set the path to the raw data
synthetic_data_dir = raw_dir+"/synthetic_data.json"
data_overview_dir = raw_dir+"/data_overview.json"
data_dictionary_dir = raw_dir+"/data_dictionary.json"
data_anomaly_dir = raw_dir+"/data_anomaly.json"

#assign the raw data to variables 
data_overview = spark.read.format("json").load(data_overview_dir)
data_dictionary = spark.read.format("json").load(data_dictionary_dir)
data_anomaly = spark.read.format("json").load(data_anomaly_dir)
synthetic_data = spark.read.format("json").load(synthetic_data_dir)


In [0]:
display(data_overview)

Create tables

In [0]:
synthetic_data = synthetic_data.withColumnRenamed("_id","id").withColumnRenamed("Transaction Timestamp","timestamp").withColumnRenamed("Usage Amount","usage_amount").withColumnRenamed("User ID","user_id").withColumnRenamed("Account Number","account_number").withColumnRenamed("Registration Date","registration_date").withColumnRenamed("Status","status").withColumnRenamed("Email","email").withColumnRenamed("Name","name").withColumnRenamed("Phone","phone").drop("id")
display(synthetic_data)

Create delta tables for control_datalake

In [0]:
catalog = "raw_control_datalake"
database = "dev"
spark.sql(f"create database if not exists {catalog}.{database}")
spark.sql(f"use {catalog}.{database}")
#write the data to the database
data_overview.write.mode("overwrite").saveAsTable("raw_data_overview")
data_dictionary.write.mode("overwrite").saveAsTable("raw_data_dictionary")
data_anomaly.write.mode("overwrite").saveAsTable("raw_data_anomaly")
synthetic_data.write.mode("overwrite").saveAsTable("raw_synthetic_data")



create control logic table

In [0]:
%sql
USE raw_control_datalake.dev;
  

In [0]:
df =spark.read.table("raw_data_anomaly")
display(df)

In [0]:
catalog = "raw_control_datalake"
database = "dev"
spark.sql(f"use {catalog}.{database}")


Create control logic

In [0]:
synthetic_df = spark.read.table("raw_synthetic_data").filter("status='Suspended'")
display(synthetic_df)


In [0]:
# Convert it into python

control_logic = f"""SELECT *
                    FROM raw_control_datalake.dev.raw_synthetic_data
                    WHERE status = 'Suspended';
                    """

In [0]:
spark.sql(control_logic).display()

Create a control logic table (USE upserting for control logic appending)

In [0]:
control_logic = spark.createDataFrame([
    (1, control_logic, "Check suspended customers", "ACTIVE")
], ["reference_number","control_logic","control_logic_description","control_logic_status"])


Adding a timestamp field for audit reasons

In [0]:
control_logic = control_logic.withColumn("created_timestamp", F.current_timestamp())

create Temp table

In [0]:
control_logic.createOrReplaceTempView("control_logic_temp")


In [0]:
%sql
SELECT *
FROM control_logic_temp

Upserting will be done on the incremental dev part

In [0]:
'''spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{database}.raw_control_logic
(
  Reference_number long,
  Control_logic string,
  Control_logic_description string,
  Control_logic_status string,
  Created_timestamp timestamp
 """)'''

In [0]:

'''
spark.sql(f"""
MERGE INTO {catalog}.{database}.raw_control_logic a
USING control_logic_temp b
ON a.mac_address=b.mac_address AND a.gym=b.gym AND a.login=b.login
WHEN NOT MATCHED THEN INSERT *
""")'''

In [0]:
display(control_logic)

In [0]:

catalog ='raw_control_datalake'
database = 'dev'
control_logic.write.mode("overwrite").saveAsTable(f"{catalog}.{database}.raw_control_logic")